In [ ]:
# ============================================================
# 🍃 Multimodal Fusion: Digital (ResNet50) + Thermal (DenseNet121)
# ============================================================
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Concatenate, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# -----------------------------
# Load pretrained .h5 models
# -----------------------------
digital_model = load_model("digital_resnet50_final.h5")
thermal_model = load_model("best_densenet_thermal_guava.h5")

# Freeze pretrained weights
for layer in digital_model.layers:
    layer.trainable = False
for layer in thermal_model.layers:
    layer.trainable = False

# -----------------------------
# Define Input Layers
# -----------------------------
input_digital = Input(shape=(224, 224, 3), name="digital_input")
input_thermal = Input(shape=(224, 224, 3), name="thermal_input")

# Get feature outputs
feat_digital = digital_model(input_digital)
feat_thermal = thermal_model(input_thermal)

# -----------------------------
# Feature Fusion (Concatenate)
# -----------------------------
fusion = Concatenate()([feat_digital, feat_thermal])
fusion = Dense(512, activation='relu')(fusion)
fusion = Dropout(0.4)(fusion)
output = Dense(3, activation='softmax')(fusion)  # 3 classes

# -----------------------------
# Final Fusion Model
# -----------------------------
fusion_model = Model(inputs=[input_digital, input_thermal], outputs=output)
fusion_model.summary()

# -----------------------------
# Compile the model
# -----------------------------
fusion_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# -----------------------------
# Callbacks
# -----------------------------
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2),
    ModelCheckpoint("fusion_best_model.h5", monitor='val_accuracy', save_best_only=True)
]

# -----------------------------
# Train the model
# -----------------------------
history = fusion_model.fit(
    [X_train_digital, X_train_thermal], y_train,
    validation_data=([X_val_digital, X_val_thermal], y_val),
    epochs=50,
    batch_size=32,
    callbacks=callbacks
)

# -----------------------------
# Save Final Model
# -----------------------------
fusion_model.save("final_multimodal_fusion_model.h5")

In [2]:
pip install tensorflow

  Using cached tensorflow-2.20.0-cp311-cp311-win_amd64.whl.metadata (4.6 kB)
  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.9.23-py2.py3-none-any.whl.metadata (875 bytes)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-6.33.0-cp310-abi3-win_amd64.whl.metadata (593 bytes)
  Using cached termcolor-3.2.0-py3-none-any.whl.metadata (6.4 kB)
  Using cached grpcio-1.76.0-cp311-cp311-win_amd64.whl.metadata (3.8 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached keras-3.12.0-py3-none-any.whl.metadata (5.9 kB)
  Using cached h5py-3.15.1-cp311-cp311-win_amd64.whl.metadata (3.1 kB)
  Using cached m

In [3]:
# ============================================================
# 🍃 Multimodal Fusion: Digital (ResNet50) + Thermal (DenseNet121)
# ============================================================
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Dense, Dropout, Concatenate, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import numpy as np

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

# -----------------------------
# Load pretrained .h5 models
# -----------------------------
print("\n[1] Loading pretrained models...")
try:
    digital_model = load_model("digital_resnet50_final.h5", compile=False)
    print("✓ Digital model loaded")
except Exception as e:
    print(f"✗ Error loading digital model: {e}")
    exit(1)

try:
    thermal_model = load_model("best_densenet_thermal_guava.h5", compile=False)
    print("✓ Thermal model loaded")
except Exception as e:
    print(f"✗ Error loading thermal model: {e}")
    exit(1)

# -----------------------------
# Remove classification heads and extract features
# -----------------------------
print("\n[2] Extracting feature layers...")

# For ResNet50 - remove last layers to get features
# Find the layer before final dense/classification
digital_feature_layer = None
for layer in reversed(digital_model.layers):
    if 'global' in layer.name.lower() or 'pool' in layer.name.lower():
        digital_feature_layer = layer.output
        break
    if 'dense' not in layer.name.lower() and len(layer.output_shape) > 2:
        digital_feature_layer = layer.output
        break

if digital_feature_layer is None:
    # If not found, use layer before last few layers
    digital_feature_layer = digital_model.layers[-3].output

print(f"Digital feature shape: {digital_feature_layer.shape}")

# For DenseNet - remove last layers to get features
thermal_feature_layer = None
for layer in reversed(thermal_model.layers):
    if 'global' in layer.name.lower() or 'pool' in layer.name.lower():
        thermal_feature_layer = layer.output
        break
    if 'dense' not in layer.name.lower() and len(layer.output_shape) > 2:
        thermal_feature_layer = layer.output
        break

if thermal_feature_layer is None:
    thermal_feature_layer = thermal_model.layers[-3].output

print(f"Thermal feature shape: {thermal_feature_layer.shape}")

# Create feature extraction models
digital_feature_extractor = Model(inputs=digital_model.input, outputs=digital_feature_layer)
thermal_feature_extractor = Model(inputs=thermal_model.input, outputs=thermal_feature_layer)

# Freeze pretrained weights
for layer in digital_feature_extractor.layers:
    layer.trainable = False
for layer in thermal_feature_extractor.layers:
    layer.trainable = False

print("✓ Feature extractors created and frozen")

# -----------------------------
# Define Input Layers
# -----------------------------
print("\n[3] Building fusion architecture...")
input_digital = Input(shape=(224, 224, 3), name="digital_input")
input_thermal = Input(shape=(224, 224, 3), name="thermal_input")

# Get feature outputs
feat_digital = digital_feature_extractor(input_digital)
feat_thermal = thermal_feature_extractor(input_thermal)

# Flatten if needed
if len(feat_digital.shape) > 2:
    from tensorflow.keras.layers import GlobalAveragePooling2D, Flatten
    feat_digital = GlobalAveragePooling2D()(feat_digital)
if len(feat_thermal.shape) > 2:
    feat_thermal = GlobalAveragePooling2D()(feat_thermal)

print(f"Digital feature after pooling: {feat_digital.shape}")
print(f"Thermal feature after pooling: {feat_thermal.shape}")

# -----------------------------
# Feature Fusion (Concatenate)
# -----------------------------
fusion = Concatenate()([feat_digital, feat_thermal])
fusion = Dense(512, activation='relu', name='fusion_dense1')(fusion)
fusion = Dropout(0.4)(fusion)
fusion = Dense(256, activation='relu', name='fusion_dense2')(fusion)
fusion = Dropout(0.3)(fusion)
output = Dense(3, activation='softmax', name='output')(fusion)  # 3 classes

# -----------------------------
# Final Fusion Model
# -----------------------------
fusion_model = Model(inputs=[input_digital, input_thermal], outputs=output)
print("\n[4] Model Architecture:")
fusion_model.summary()

# -----------------------------
# Compile the model
# -----------------------------
print("\n[5] Compiling model...")
fusion_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print("✓ Model compiled")

# -----------------------------
# Check data shapes (IMPORTANT)
# -----------------------------
print("\n[6] Checking data...")
print("Before training, ensure your data is loaded:")
print("Expected shapes:")
print("  X_train_digital: (n_samples, 224, 224, 3)")
print("  X_train_thermal: (n_samples, 224, 224, 3)")
print("  y_train: (n_samples, 3) - one-hot encoded")
print("\nExample data loading:")
print("""
# Load your data here
X_train_digital = np.load('path/to/digital_train.npy')  # or load images
X_train_thermal = np.load('path/to/thermal_train.npy')
y_train = np.load('path/to/labels_train.npy')

X_val_digital = np.load('path/to/digital_val.npy')
X_val_thermal = np.load('path/to/thermal_val.npy')
y_val = np.load('path/to/labels_val.npy')

# Normalize images to [0, 1]
X_train_digital = X_train_digital / 255.0
X_train_thermal = X_train_thermal / 255.0
X_val_digital = X_val_digital / 255.0
X_val_thermal = X_val_thermal / 255.0

# Check shapes
print(f"X_train_digital shape: {X_train_digital.shape}")
print(f"X_train_thermal shape: {X_train_thermal.shape}")
print(f"y_train shape: {y_train.shape}")
""")

# -----------------------------
# Callbacks
# -----------------------------
print("\n[7] Setting up callbacks...")
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        "fusion_best_model.h5",
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]
print("✓ Callbacks configured")

# -----------------------------
# Train the model
# -----------------------------
print("\n[8] Starting training...")
print("=" * 60)

# UNCOMMENT AFTER LOADING YOUR DATA
"""
try:
    history = fusion_model.fit(
        [X_train_digital, X_train_thermal],
        y_train,
        validation_data=([X_val_digital, X_val_thermal], y_val),
        epochs=50,
        batch_size=32,
        callbacks=callbacks,
        verbose=1
    )
    
    print("\n✓ Training completed!")
    
    # Save Final Model
    fusion_model.save("final_multimodal_fusion_model.h5")
    print("✓ Final model saved")
    
except Exception as e:
    print(f"\n✗ Training error: {e}")
    import traceback
    traceback.print_exc()
"""

print("\n" + "=" * 60)
print("IMPORTANT: Uncomment the training section after loading data!")
print("=" * 60)

TensorFlow Version: 2.20.0
GPU Available: []

[1] Loading pretrained models...
✗ Error loading digital model: [Errno 2] Unable to synchronously open file (unable to open file: name = 'digital_resnet50_final.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)
✗ Error loading thermal model: [Errno 2] Unable to synchronously open file (unable to open file: name = 'best_densenet_thermal_guava.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

[2] Extracting feature layers...


NameError: name 'digital_model' is not defined

In [6]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model, Model
import numpy as np

class MultimodalFusion:
    """
    Multimodal fusion for combining digital (ResNet50) and thermal (DenseNet) image models
    """
    
    def __init__(self, digital_model_path, thermal_model_path, num_classes):
        """
        Initialize the multimodal fusion system
        
        Args:
            digital_model_path: Path to ResNet50 .h5 model
            thermal_model_path: Path to DenseNet .h5 model
            num_classes: Number of output classes
        """
        self.digital_model_path = digital_model_path
        self.thermal_model_path = thermal_model_path
        self.num_classes = num_classes
        self.fusion_model = None
        
    def load_pretrained_models(self):
        """Load the pretrained ResNet50 and DenseNet models"""
        print("Loading digital image model (ResNet50)...")
        self.digital_model = load_model(self.digital_model_path)
        
        print("Loading thermal image model (DenseNet)...")
        self.thermal_model = load_model(self.thermal_model_path)
        
        # Remove the final classification layers to extract features
        self.digital_feature_extractor = Model(
            inputs=self.digital_model.input,
            outputs=self.digital_model.layers[-2].output  # Before final dense layer
        )
        
        self.thermal_feature_extractor = Model(
            inputs=self.thermal_model.input,
            outputs=self.thermal_model.layers[-2].output
        )
        
        # Freeze the pretrained weights
        for layer in self.digital_feature_extractor.layers:
            layer.trainable = False
        for layer in self.thermal_feature_extractor.layers:
            layer.trainable = False
            
        print("Models loaded successfully!")
        
    def create_early_fusion_model(self, input_shape_digital, input_shape_thermal):
        """
        Early fusion: Concatenate inputs before processing
        
        Args:
            input_shape_digital: Shape of digital images (height, width, channels)
            input_shape_thermal: Shape of thermal images (height, width, channels)
        """
        # Input layers
        digital_input = layers.Input(shape=input_shape_digital, name='digital_input')
        thermal_input = layers.Input(shape=input_shape_thermal, name='thermal_input')
        
        # Concatenate inputs
        concatenated = layers.Concatenate(axis=-1)([digital_input, thermal_input])
        
        # Shared processing
        x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(concatenated)
        x = layers.MaxPooling2D((2, 2))(x)
        x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
        x = layers.GlobalAveragePooling2D()(x)
        
        # Classification head
        x = layers.Dense(256, activation='relu')(x)
        x = layers.Dropout(0.5)(x)
        output = layers.Dense(self.num_classes, activation='softmax', name='output')(x)
        
        self.fusion_model = Model(inputs=[digital_input, thermal_input], outputs=output)
        return self.fusion_model
    
    def create_late_fusion_model(self, fusion_strategy='concatenate'):
        """
        Late fusion: Extract features separately then combine
        
        Args:
            fusion_strategy: 'concatenate', 'average', 'maximum', or 'attention'
        """
        # Input layers
        digital_input = layers.Input(shape=self.digital_feature_extractor.input_shape[1:], 
                                     name='digital_input')
        thermal_input = layers.Input(shape=self.thermal_feature_extractor.input_shape[1:], 
                                     name='thermal_input')
        
        # Extract features from pretrained models
        digital_features = self.digital_feature_extractor(digital_input)
        thermal_features = self.thermal_feature_extractor(thermal_input)
        
        # Fusion strategies
        if fusion_strategy == 'concatenate':
            fused = layers.Concatenate()([digital_features, thermal_features])
        elif fusion_strategy == 'average':
            fused = layers.Average()([digital_features, thermal_features])
        elif fusion_strategy == 'maximum':
            fused = layers.Maximum()([digital_features, thermal_features])
        elif fusion_strategy == 'attention':
            # Attention-based fusion
            attention_digital = layers.Dense(1, activation='sigmoid')(digital_features)
            attention_thermal = layers.Dense(1, activation='sigmoid')(thermal_features)
            
            weighted_digital = layers.Multiply()([digital_features, attention_digital])
            weighted_thermal = layers.Multiply()([thermal_features, attention_thermal])
            
            fused = layers.Add()([weighted_digital, weighted_thermal])
        else:
            raise ValueError(f"Unknown fusion strategy: {fusion_strategy}")
        
        # Classification head
        x = layers.Dense(512, activation='relu')(fused)
        x = layers.Dropout(0.5)(x)
        x = layers.Dense(256, activation='relu')(x)
        x = layers.Dropout(0.3)(x)
        output = layers.Dense(self.num_classes, activation='softmax', name='output')(x)
        
        self.fusion_model = Model(inputs=[digital_input, thermal_input], outputs=output)
        return self.fusion_model
    
    def create_intermediate_fusion_model(self):
        """
        Intermediate fusion: Combine features at multiple levels
        """
        # Get intermediate layers from both models
        digital_input = layers.Input(shape=self.digital_feature_extractor.input_shape[1:], 
                                     name='digital_input')
        thermal_input = layers.Input(shape=self.thermal_feature_extractor.input_shape[1:], 
                                     name='thermal_input')
        
        # Extract features
        digital_features = self.digital_feature_extractor(digital_input)
        thermal_features = self.thermal_feature_extractor(thermal_input)
        
        # Multi-level fusion
        concat_features = layers.Concatenate()([digital_features, thermal_features])
        
        x = layers.Dense(1024, activation='relu')(concat_features)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.5)(x)
        
        x = layers.Dense(512, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.3)(x)
        
        output = layers.Dense(self.num_classes, activation='softmax')(x)
        
        self.fusion_model = Model(inputs=[digital_input, thermal_input], outputs=output)
        return self.fusion_model
    
    def compile_model(self, learning_rate=0.0001):
        """Compile the fusion model"""
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
        self.fusion_model.compile(
            optimizer=optimizer,
            loss='categorical_crossentropy',
            metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall()]
        )
        
    def train(self, digital_train, thermal_train, y_train, 
              digital_val, thermal_val, y_val,
              epochs=50, batch_size=32):
        """
        Train the fusion model
        
        Args:
            digital_train: Training digital images
            thermal_train: Training thermal images
            y_train: Training labels (one-hot encoded)
            digital_val: Validation digital images
            thermal_val: Validation thermal images
            y_val: Validation labels
            epochs: Number of training epochs
            batch_size: Batch size
        """
        callbacks = [
            keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7),
            keras.callbacks.ModelCheckpoint('best_fusion_model.h5', save_best_only=True, monitor='val_accuracy')
        ]
        
        history = self.fusion_model.fit(
            [digital_train, thermal_train],
            y_train,
            validation_data=([digital_val, thermal_val], y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=callbacks,
            verbose=1
        )
        
        return history
    
    def predict(self, digital_images, thermal_images):
        """Make predictions on new data"""
        return self.fusion_model.predict([digital_images, thermal_images])
    
    def evaluate(self, digital_test, thermal_test, y_test):
        """Evaluate the model on test data"""
        return self.fusion_model.evaluate([digital_test, thermal_test], y_test)
    
    def save_model(self, filepath):
        """Save the fusion model"""
        self.fusion_model.save(filepath)
        print(f"Fusion model saved to {filepath}")


# Example usage
if __name__ == "__main__":
    # Configuration
    DIGITAL_MODEL_PATH = r"C:\Users\cl502_15\Downloads\Mtech Major Project\Mtech Major Project\DeepLearningModels\digital_resnet50_final.h5"
    THERMAL_MODEL_PATH = r"C:\Users\cl502_15\Downloads\Mtech Major Project\Mtech Major Project\DeepLearningModels\densenet121_thermal_guava_final.h5"
    NUM_CLASSES = 3  # Change based on your dataset
    
    # Initialize fusion system
    fusion_system = MultimodalFusion(
        digital_model_path=DIGITAL_MODEL_PATH,
        thermal_model_path=THERMAL_MODEL_PATH,
        num_classes=NUM_CLASSES
    )
    
    # Load pretrained models
    fusion_system.load_pretrained_models()
    
    # Create fusion model (choose one strategy)
    # Option 1: Late Fusion with concatenation
    model = fusion_system.create_late_fusion_model(fusion_strategy='concatenate')
    
    # Option 2: Late Fusion with attention
    # model = fusion_system.create_late_fusion_model(fusion_strategy='attention')
    
    # Option 3: Intermediate Fusion
    # model = fusion_system.create_intermediate_fusion_model()
    
    # Compile model
    fusion_system.compile_model(learning_rate=0.0001)
    
    # Print model summary
    model.summary()
    
    # Prepare your data (example)
    digital_train, thermal_train, y_train = load_your_training_data()
    digital_val, thermal_val, y_val = load_your_validation_data()
    
    # Train the model
    history = fusion_system.train(
        digital_train, thermal_train, y_train,
        digital_val, thermal_val, y_val,
        epochs=50,
        batch_size=32
    )
    
    # Save the trained fusion model
    fusion_system.save_model('multimodal_fusion_model.h5')
    
    # Make predictions
    predictions = fusion_system.predict(digital_test, thermal_test)
    
    # Evaluate
    results = fusion_system.evaluate(digital_test, thermal_test, y_test)
    print(f"Test Accuracy: {results[1]:.4f}")

Loading digital image model (ResNet50)...


Loading thermal image model (DenseNet)...


Models loaded successfully!


Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ digital_input       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ thermal_input       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_6        │ (None, 2048)      │ 23,587,712 │ digital_input[0]… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_7        │ (None, 1024)      │  7,037,504 │ thermal_input[0]… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 3072)      │          0 │ functional_6[0][… │
│ (Concatenate)       │                   │            │ functional_7[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 512)       │  1,573,376 │ concatenate_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 512)       │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 256)       │    131,328 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 256)       │          0 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 3)         │        771 │ dropout_5[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 32,330,691 (123.33 MB)

 Trainable params: 1,705,475 (6.51 MB)

 Non-trainable params: 30,625,216 (116.83 MB)

NameError: name 'load_your_training_data' is not defined

In [10]:
pip install scikit-learn

  Using cached scikit_learn-1.7.2-cp311-cp311-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.16.3-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.5.2-py3-none-any.whl.metadata (5.6 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.7.2-cp311-cp311-win_amd64.whl (8.9 MB)
Using cached joblib-1.5.2-py3-none-any.whl (308 kB)
Using cached scipy-1.16.3-cp311-cp311-win_amd64.whl (38.7 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ------------------

In [12]:
pip install matplotlib

  Using cached matplotlib-3.10.7-cp311-cp311-win_amd64.whl.metadata (11 kB)
  Using cached contourpy-1.3.3-cp311-cp311-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.60.1-cp311-cp311-win_amd64.whl.metadata (114 kB)
  Using cached kiwisolver-1.4.9-cp311-cp311-win_amd64.whl.metadata (6.4 kB)
  Using cached pyparsing-3.2.5-py3-none-any.whl.metadata (5.0 kB)
Using cached matplotlib-3.10.7-cp311-cp311-win_amd64.whl (8.1 MB)
Using cached contourpy-1.3.3-cp311-cp311-win_amd64.whl (225 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.60.1-cp311-cp311-win_amd64.whl (2.3 MB)
Using cached kiwisolver-1.4.9-cp311-cp311-win_amd64.whl (73 kB)
Using cached pyparsing-3.2.5-py3-none-any.whl (113 kB)

   ---------------------------------------- 0/6 [pyparsing]
   ------------- -------------------------- 2/6 [fonttools]
   ------------- -------------------------- 2/6 [fonttools]
   ------------- --

In [14]:
pip install seaborn

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached pandas-2.3.3-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Using cached pandas-2.3.3-cp311-cp311-win_amd64.whl (11.3 MB)
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)

   ---------------------------------------- 0/4 [pytz]
   ---------------------------------------- 0/4 [pytz]
   ---------------------------------------- 0/4 [pytz]
   ---------------------------------------- 0/4 [pytz]
   ---------------------------------------- 0/4 [pytz]
   ---------------------------------------- 0/4 [pytz]
   ---------------------------------------- 0/4 [pytz]
   ---------------------------------------- 0/4 [pytz]
   ---------- ----------------------------- 1/4 [tzdata]
   --

In [10]:
# ============================================================
# Multimodal Fusion: Digital + Thermal (Memory-based approach)
# ============================================================
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import Dense, Dropout, Concatenate, GlobalAveragePooling2D, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.utils import to_categorical
import numpy as np
import os
from glob import glob
from sklearn.model_selection import train_test_split

# ---------------------- CONFIG ----------------------
digital_model_path = r"C:\Users\cl502_15\Downloads\Mtech Major Project\Mtech Major Project\DeepLearningModels\digital_resnet50_final.h5"
thermal_model_path = r"C:\Users\cl502_15\Downloads\Mtech Major Project\Mtech Major Project\DeepLearningModels\densenet121_thermal_guava_final.h5"

digital_train_dir = r"C:\Users\cl502_15\Downloads\Mtech Major Project\Mtech Major Project\Dataset\Maturity\train test val split for digital\train"
thermal_train_dir = r"C:\Users\cl502_15\Downloads\Mtech Major Project\Mtech Major Project\Dataset\Maturity\train test val split  thermal\train"
digital_val_dir = r"C:\Users\cl502_15\Downloads\Mtech Major Project\Mtech Major Project\Dataset\Maturity\train test val split for digital\val"
thermal_val_dir = r"C:\Users\cl502_15\Downloads\Mtech Major Project\Mtech Major Project\Dataset\Maturity\train test val split  thermal\val"

BATCH_SIZE = 16
IMAGE_SIZE = (224, 224)
EPOCHS = 50
LR = 1e-4

# ============================================================
# STEP 1: Load images into memory
# ============================================================
def load_images_from_directory(digital_dir, thermal_dir):
    """Load all images from directories into arrays"""
    print(f"\nLoading images from {os.path.basename(digital_dir)}...")
    
    classes = sorted(os.listdir(digital_dir))
    class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
    
    digital_images = []
    thermal_images = []
    labels = []
    
    for class_name in classes:
        digital_class_dir = os.path.join(digital_dir, class_name)
        thermal_class_dir = os.path.join(thermal_dir, class_name)
        
        if not os.path.isdir(digital_class_dir) or not os.path.isdir(thermal_class_dir):
            continue
        
        digital_files = sorted(glob(os.path.join(digital_class_dir, '*.*')))
        thermal_files = sorted(glob(os.path.join(thermal_class_dir, '*.*')))
        
        print(f"  Class '{class_name}': {len(digital_files)} digital, {len(thermal_files)} thermal")
        
        # Ensure same number of files
        min_files = min(len(digital_files), len(thermal_files))
        
        for i in range(min_files):
            # Load digital image
            img_d = load_img(digital_files[i], target_size=IMAGE_SIZE)
            img_d = img_to_array(img_d) / 255.0
            digital_images.append(img_d)
            
            # Load thermal image
            img_t = load_img(thermal_files[i], target_size=IMAGE_SIZE)
            img_t = img_to_array(img_t) / 255.0
            thermal_images.append(img_t)
            
            # Label
            labels.append(class_to_idx[class_name])
    
    digital_images = np.array(digital_images, dtype='float32')
    thermal_images = np.array(thermal_images, dtype='float32')
    labels = np.array(labels)
    
    print(f"  Total loaded: {len(labels)} image pairs")
    print(f"  Digital shape: {digital_images.shape}")
    print(f"  Thermal shape: {thermal_images.shape}")
    
    return digital_images, thermal_images, labels, len(classes)

# Load training data
X_train_digital, X_train_thermal, y_train, NUM_CLASSES = load_images_from_directory(
    digital_train_dir, thermal_train_dir
)

# Load validation data
X_val_digital, X_val_thermal, y_val, _ = load_images_from_directory(
    digital_val_dir, thermal_val_dir
)

# Convert labels to categorical
y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_val_cat = to_categorical(y_val, NUM_CLASSES)

print(f"\nDataset summary:")
print(f"  Training samples: {len(y_train)}")
print(f"  Validation samples: {len(y_val)}")
print(f"  Number of classes: {NUM_CLASSES}")

# ============================================================
# STEP 2: Load pretrained models and create feature extractors
# ============================================================
print("\n" + "="*60)
print("Loading pretrained models...")
print("="*60)

digital_model = load_model(digital_model_path, compile=False)
thermal_model = load_model(thermal_model_path, compile=False)

print(f"\nDigital model: {len(digital_model.layers)} layers")
print(f"Thermal model: {len(thermal_model.layers)} layers")

# Find the feature layer (last non-Dense layer)
def create_feature_model(base_model, model_name):
    """Create feature extractor by removing classification head"""
    print(f"\nProcessing {model_name}:")
    
    # Find last layer that's not Dense or Dropout
    feature_layer_idx = None
    for i in range(len(base_model.layers) - 1, -1, -1):
        layer = base_model.layers[i]
        if not isinstance(layer, (Dense, Dropout)):
            feature_layer_idx = i
            print(f"  Selected layer {i}: {layer.name} ({type(layer).__name__})")
            break
    
    if feature_layer_idx is None:
        raise ValueError(f"Could not find feature layer in {model_name}")
    
    # Create feature extraction model
    feature_output = base_model.layers[feature_layer_idx].output
    
    # Add pooling if needed
    if len(feature_output.shape) == 4:
        print(f"  Adding GlobalAveragePooling2D")
        feature_output = GlobalAveragePooling2D()(feature_output)
    
    feature_model = Model(
        inputs=base_model.input,
        outputs=feature_output
    )
    
    feature_model.trainable = False
    print(f"  Output shape: {feature_model.output.shape}")
    
    return feature_model

digital_features = create_feature_model(digital_model, "Digital")
thermal_features = create_feature_model(thermal_model, "Thermal")

# ============================================================
# STEP 3: Extract features from images
# ============================================================
print("\n" + "="*60)
print("Extracting features from images...")
print("="*60)

print("\nExtracting training features...")
train_digital_feat = digital_features.predict(X_train_digital, batch_size=BATCH_SIZE, verbose=1)
train_thermal_feat = thermal_features.predict(X_train_thermal, batch_size=BATCH_SIZE, verbose=1)

print("\nExtracting validation features...")
val_digital_feat = digital_features.predict(X_val_digital, batch_size=BATCH_SIZE, verbose=1)
val_thermal_feat = thermal_features.predict(X_val_thermal, batch_size=BATCH_SIZE, verbose=1)

print(f"\nFeature extraction complete:")
print(f"  Train digital features: {train_digital_feat.shape}")
print(f"  Train thermal features: {train_thermal_feat.shape}")
print(f"  Val digital features: {val_digital_feat.shape}")
print(f"  Val thermal features: {val_thermal_feat.shape}")

# Free memory
del digital_model, thermal_model, digital_features, thermal_features
del X_train_digital, X_train_thermal, X_val_digital, X_val_thermal
import gc
gc.collect()

# ============================================================
# STEP 4: Build fusion model
# ============================================================
print("\n" + "="*60)
print("Building fusion classifier...")
print("="*60)

# Get feature dimensions
digital_feat_dim = train_digital_feat.shape[1]
thermal_feat_dim = train_thermal_feat.shape[1]

print(f"\nInput dimensions:")
print(f"  Digital features: {digital_feat_dim}")
print(f"  Thermal features: {thermal_feat_dim}")

# Build model
input_digital = Input(shape=(digital_feat_dim,), name='digital_features')
input_thermal = Input(shape=(thermal_feat_dim,), name='thermal_features')

# Concatenate
combined = Concatenate()([input_digital, input_thermal])

# Classification head
x = Dense(512, activation='relu')(combined)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(NUM_CLASSES, activation='softmax')(x)

# Create model
fusion_model = Model(
    inputs=[input_digital, input_thermal],
    outputs=output,
    name='fusion_classifier'
)

# Compile
fusion_model.compile(
    optimizer=Adam(learning_rate=LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\nModel architecture:")
fusion_model.summary()

# ============================================================
# STEP 5: Setup callbacks
# ============================================================
os.makedirs('models', exist_ok=True)

callbacks = [
    ModelCheckpoint(
        'models/best_fusion_model.h5',
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
]

# ============================================================
# STEP 6: Train fusion model
# ============================================================
print("\n" + "="*60)
print("Training fusion model...")
print("="*60)

history = fusion_model.fit(
    [train_digital_feat, train_thermal_feat],
    y_train_cat,
    validation_data=([val_digital_feat, val_thermal_feat], y_val_cat),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

# ============================================================
# STEP 7: Save model and results
# ============================================================
print("\n" + "="*60)
print("Saving model and results...")
print("="*60)

fusion_model.save('models/final_fusion_model.h5')
print("\nModel saved to 'models/final_fusion_model.h5'")

# Save history
import pickle
with open('models/training_history.pkl', 'wb') as f:
    pickle.dump(history.history, f)
print("Training history saved")

# ============================================================
# STEP 8: Display results
# ============================================================
print("\n" + "="*60)
print("TRAINING RESULTS")
print("="*60)

final_train_acc = history.history['accuracy'][-1]
final_train_loss = history.history['loss'][-1]
final_val_acc = history.history['val_accuracy'][-1]
final_val_loss = history.history['val_loss'][-1]

best_val_acc = max(history.history['val_accuracy'])
best_epoch = history.history['val_accuracy'].index(best_val_acc) + 1

print(f"\nFinal Metrics:")
print(f"  Training Accuracy:   {final_train_acc:.4f}")
print(f"  Training Loss:       {final_train_loss:.4f}")
print(f"  Validation Accuracy: {final_val_acc:.4f}")
print(f"  Validation Loss:     {final_val_loss:.4f}")

print(f"\nBest Performance:")
print(f"  Best Val Accuracy: {best_val_acc:.4f} at epoch {best_epoch}")

print("\n" + "="*60)
print("Training complete!")
print("="*60)


Loading images from train...
  Class 'immature': 2280 digital, 2424 thermal
  Class 'mature': 2257 digital, 2346 thermal
  Class 'semi_mature': 1994 digital, 1869 thermal
  Total loaded: 6406 image pairs
  Digital shape: (6406, 224, 224, 3)
  Thermal shape: (6406, 224, 224, 3)

Loading images from val...
  Class 'immature': 488 digital, 519 thermal
  Class 'mature': 483 digital, 502 thermal
  Class 'semi_mature': 427 digital, 400 thermal
  Total loaded: 1371 image pairs
  Digital shape: (1371, 224, 224, 3)
  Thermal shape: (1371, 224, 224, 3)

Dataset summary:
  Training samples: 6406
  Validation samples: 1371
  Number of classes: 3

Loading pretrained models...

Digital model: 178 layers
Thermal model: 430 layers

Processing Digital:
  Selected layer 175: global_average_pooling2d (GlobalAveragePooling2D)
  Output shape: (None, 2048)

Processing Thermal:
  Selected layer 427: global_average_pooling2d (GlobalAveragePooling2D)
  Output shape: (None, 1024)

Extracting features from imag

Model: "fusion_classifier"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ digital_features    │ (None, 2048)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ thermal_features    │ (None, 1024)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 3072)      │          0 │ digital_features… │
│ (Concatenate)       │                   │            │ thermal_features… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 512)       │  1,573,376 │ concatenate_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_9 (Dropout) │ (None, 512)       │          0 │ dense_13[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 256)       │    131,328 │ dropout_9[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 256)       │          0 │ dense_14[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_15 (Dense)    │ (None, 128)       │     32,896 │ dropout_10[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_11          │ (None, 128)       │          0 │ dense_15[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_16 (Dense)    │ (None, 3)         │        387 │ dropout_11[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,737,987 (6.63 MB)

 Trainable params: 1,737,987 (6.63 MB)

...

401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.8820 - loss: 0.3033 - val_accuracy: 0.9263 - val_loss: 0.2224 - learning_rate: 1.0000e-04
Epoch 19/50
399/401 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8902 - loss: 0.2800
Epoch 19: val_accuracy did not improve from 0.92633
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.8926 - loss: 0.2733 - val_accuracy: 0.9198 - val_loss: 0.2147 - learning_rate: 1.0000e-04
Epoch 20/50
399/401 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9018 - loss: 0.2560
Epoch 20: val_accuracy improved from 0.92633 to 0.93071, saving model to models/best_fusion_model.h5


401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.8963 - loss: 0.2667 - val_accuracy: 0.9307 - val_loss: 0.2082 - learning_rate: 1.0000e-04
Epoch 21/50
398/401 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9005 - loss: 0.2526
Epoch 21: val_accuracy did not improve from 0.93071
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.8984 - loss: 0.2598 - val_accuracy: 0.9147 - val_loss: 0.2302 - learning_rate: 1.0000e-04
Epoch 22/50
397/401 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8959 - loss: 0.2612
Epoch 22: val_accuracy did not improve from 0.93071
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.8990 - loss: 0.2631 - val_accuracy: 0.9154 - val_loss: 0.2049 - learning_rate: 1.0000e-04
Epoch 23/50
399/401 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9018 - loss: 0.2554
Epoch 23: val_accuracy did not improve from 0.93071
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9002 - loss: 0.2555 - val_accuracy: 0.9285 - val_loss: 0.2013 - learning_rate: 1.0000e

401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9123 - loss: 0.2279 - val_accuracy: 0.9358 - val_loss: 0.1733 - learning_rate: 1.0000e-04
Epoch 27/50
397/401 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9111 - loss: 0.2297
Epoch 27: val_accuracy improved from 0.93581 to 0.95186, saving model to models/best_fusion_model.h5


401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9126 - loss: 0.2286 - val_accuracy: 0.9519 - val_loss: 0.1522 - learning_rate: 1.0000e-04
Epoch 28/50
398/401 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9266 - loss: 0.2017
Epoch 28: val_accuracy did not improve from 0.95186
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9188 - loss: 0.2143 - val_accuracy: 0.9402 - val_loss: 0.1548 - learning_rate: 1.0000e-04
Epoch 29/50
400/401 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9254 - loss: 0.2013
Epoch 29: val_accuracy did not improve from 0.95186
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9248 - loss: 0.2014 - val_accuracy: 0.9373 - val_loss: 0.1675 - learning_rate: 1.0000e-04
Epoch 30/50
397/401 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9177 - loss: 0.2080
Epoch 30: val_accuracy did not improve from 0.95186
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9182 - loss: 0.2089 - val_accuracy: 0.9424 - val_loss: 0.1682 - learning_rate: 1.0000e


Saving model and results...

Model saved to 'models/final_fusion_model.h5'
Training history saved

TRAINING RESULTS

Final Metrics:
  Training Accuracy:   0.9397
  Training Loss:       0.1674
  Validation Accuracy: 0.9416
  Validation Loss:     0.1676

Best Performance:
  Best Val Accuracy: 0.9519 at epoch 27

Training complete!


In [11]:
print("\n" + "="*60)
print("EVALUATION ON VALIDATION SET")
print("="*60)

# Make predictions
y_pred_probs = fusion_model.predict([val_digital_feat, val_thermal_feat], batch_size=BATCH_SIZE)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_val

# Classification report
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Get class names from directory structure
class_names = sorted(os.listdir(digital_train_dir))
print(f"\nClass names: {class_names}")

print("\nClassification Report:")
print("="*60)
report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report)

# Save classification report
with open('models/classification_report.txt', 'w') as f:
    f.write("Classification Report\n")
    f.write("="*60 + "\n")
    f.write(report)
print("\nClassification report saved to 'models/classification_report.txt'")

# Confusion Matrix
print("\nGenerating confusion matrix...")
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('models/confusion_matrix.png', dpi=300, bbox_inches='tight')
print("Confusion matrix saved to 'models/confusion_matrix.png'")
plt.close()

# ============================================================
# STEP 10: Training History Visualization
# ============================================================
print("\n" + "="*60)
print("GENERATING TRAINING VISUALIZATIONS")
print("="*60)

# Create figure with 2 subplots
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Accuracy
axes[0].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].legend(loc='lower right', fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Loss
axes[1].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].legend(loc='upper right', fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('models/training_history.png', dpi=300, bbox_inches='tight')
print("\nTraining history plot saved to 'models/training_history.png'")
plt.close()

# ============================================================
# STEP 11: Per-class Accuracy Visualization
# ============================================================
print("\nGenerating per-class accuracy visualization...")

# Calculate per-class accuracy
per_class_accuracy = []
for i in range(NUM_CLASSES):
    class_mask = y_true == i
    if np.sum(class_mask) > 0:
        class_acc = np.sum((y_pred == y_true) & class_mask) / np.sum(class_mask)
        per_class_accuracy.append(class_acc)
    else:
        per_class_accuracy.append(0)

# Plot per-class accuracy
plt.figure(figsize=(10, 6))
bars = plt.bar(class_names, per_class_accuracy, color='skyblue', edgecolor='navy', alpha=0.7)
plt.title('Per-Class Accuracy', fontsize=14, fontweight='bold')
plt.xlabel('Class', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.ylim([0, 1.0])
plt.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.3f}',
             ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('models/per_class_accuracy.png', dpi=300, bbox_inches='tight')
print("Per-class accuracy plot saved to 'models/per_class_accuracy.png'")
plt.close()



EVALUATION ON VALIDATION SET
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step

Class names: ['immature', 'mature', 'semi_mature']

Classification Report:
              precision    recall  f1-score   support

    immature     0.9567    0.9508    0.9538       488
      mature     0.9470    0.9627    0.9548       483
 semi_mature     0.9519    0.9400    0.9459       400

    accuracy                         0.9519      1371
   macro avg     0.9519    0.9512    0.9515      1371
weighted avg     0.9519    0.9519    0.9518      1371


Classification report saved to 'models/classification_report.txt'

Generating confusion matrix...
Confusion matrix saved to 'models/confusion_matrix.png'

GENERATING TRAINING VISUALIZATIONS

Training history plot saved to 'models/training_history.png'

Generating per-class accuracy visualization...
Per-class accuracy plot saved to 'models/per_class_accuracy.png'
